In [1]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

# prep

In [2]:
# Data prep
query = f"""

SELECT security_isin
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state f
WHERE security_type IN ('GOVS', 'FIDE')
AND assttp_scty_issr_sector_riad = 'S1311'
AND business_date >= '2023-01-01' 
AND gnlcoll = 'SPEC'
GROUP BY security_isin
HAVING COUNT(DISTINCT security_type) = 2

"""

govs = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_47732\2371169088.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  govs = pd.read_sql_query(query, cnxn)


In [3]:
unique_govs = tuple(govs['security_isin'])

In [4]:
ust = pd.read_csv('Data\\TreasuryCusip.csv')

In [5]:
treasuries = tuple(ust['ISIN'].unique())

In [6]:
# Data prep
query = f"""

SELECT
  CASE WHEN borrower_id < lender_id THEN borrower_id ELSE lender_id END AS a,
  CASE WHEN borrower_id < lender_id THEN lender_id ELSE borrower_id END AS b
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state
WHERE intragroup = 1
GROUP BY
  CASE WHEN borrower_id < lender_id THEN borrower_id ELSE lender_id END,
  CASE WHEN borrower_id < lender_id THEN lender_id ELSE borrower_id END;


"""

df_edges = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_47732\143588195.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_edges = pd.read_sql_query(query, cnxn)


In [7]:
import pandas as pd

# edges: columns ['a','b']
edges = df_edges.values.tolist()

parent = {}
rank = {}

def find(x):
    if parent.setdefault(x, x) != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    rx, ry = find(x), find(y)
    if rx == ry: return
    rxr, ryr = rank.setdefault(rx,0), rank.setdefault(ry,0)
    if rxr < ryr:
        parent[rx] = ry
    elif rxr > ryr:
        parent[ry] = rx
    else:
        parent[ry] = rx
        rank[rx] = rxr + 1

for a,b in edges:
    union(a,b)

# canonical id = min-LEI per component (stable label)
from collections import defaultdict
comps = defaultdict(list)
for x in parent.keys():
    comps[find(x)].append(x)

canon = {rep: min(members) for rep, members in comps.items()}
mapping = []
for x in parent.keys():
    rep = find(x)
    mapping.append((x, canon[rep]))

df_mapping_cc = pd.DataFrame(mapping, columns=["lei","group_id_synth"]).sort_values(["group_id_synth","lei"])


In [8]:
foreign_bg = ('0W1U67PTV5WY3WYWKD79',
'2138008P9NOMBRMROI73',
'213800A9GT65GAES2V60',
'213800G8QEXN34A2YG53',
'213800GVD8L87R18CT98',
'213800IBT39XQ9C4CP71',
'213800NW35DTWHTMX505',
'213800RZ3GCUXMBGYN59',
'2IGI19DL77OX0HC3ZE78',
'4PQUHN3JPFGFNF3BB653',
'4ZHCHI4KYZG2WVRT8631',
'5299007QVIQ7IO64NX37',
'5493000IQQ05Y25L0V92',
'5493000YPN33HF74SN02',
'54930010P7BUGOECPI58',
'5493002XYZZ0CGQ6CB58',
'54930040QPHHWGT1J432',
'54930050SE0SM7CM2G07',
'5493006PWI2H6PX25403',
'5493007VSMFZCPV1NB83',
'5493008GNQHVI377MY19',
'5493009NLZXZGJDOPC94',
'549300BKWHXYEXPV0328',
'549300FH0WJAPEHTIQ77',
'549300HU9EWFS3CNX640',
'549300KP56LL8NKKFL47',
'549300RSY622D5TQWS42',
'549300SXSTGQY3EA1B18',
'549300WDT1HWUMTUW770',
'571474TGEMMWANRLN572',
'724500AAT1DK36059L16',
'9676007O0UF5YB3QPR03',
'BWS7DNS2Z4NPKPNYKL75',
'HV5W8PGLJ127N2SFSM23'
)

In [9]:
euro_area_bg = ('0IKLU6X1B10WK7X42C15',
'1VUV7VQFKUOQSJ21A208',
'2138001YBO7CJNQOEE37',
'21380027LW8AF6I1WA03',
'2138003Z5ZVN16GFYV70',
'2138004VZX8CSGPTDX68',
'21380073P7J4PAD91E29',
'213800AGKVL18YKQSV51',
'213800BWHAS44J2C1B28',
'213800D4LHBCXXEEE235',
'213800DBQIB6VBNU5C64',
'213800FKGFR7Q2ACLS83',
'213800G63T4ER4MSVR22',
'213800HV6TP2I5A6MW58',
'213800I92TAU7I3FP232',
'213800KGF4EFNUQKAT69',
'213800OOQOSULB37T658',
'213800ZIGVOZ992FNQ85',
'222100D7H9VRJEH7DU25',
'222100M2PU043YB7YQ06',
'23B6332KMR0JIZLJG565',
'2534006G7F7F1TFC9T77',
'2534006HF1L4YF10UD91',
'253400N2R0RF14JQ0060',
'2549002MVYWWDX54IB83',
'259400QHDOZWMJ103294',
'259400YLRTOBISHBVX41',
'2W8N8UU78PMDQKZENC08',
'31570010000000036567',
'315700GBLUBZ50S45F53',
'315700GXRKZ452JF2U13',
'3M5E1GQGKL17HI6CPN30',
'3U8WV1YX2VMUHH7Z1Q21',
'48510000156ESYOBV122',
'52965FONQ5NZKP0WZL45',
'52990002O5KK6XOGJ020',
'52990010C4NK412ZL440',
'5299004TE2DYMKEAM814',
'5299005UJX6K7BQKV086',
'52990080NNXXLC14OC65',
'529900AQBND3S6YJLY83',
'529900C214QOT3ZYD838',
'529900C4RSSBWXBSY931',
'529900E1WHT64CB20277',
'529900FWTU88V844B672',
'529900GGYMNGRQTDOO93',
'529900K16YGKC8BES892',
'529900T32UL0CP1FZA06',
'529900UKZBMDBDZIXD62',
'529900VA5CNBWXAONR25',
'549300298FD7AS4PPU70',
'54930056IRBXK0Q1FP96',
'549300685QG7DJS55M76',
'5493007RT80TMKY7TO30',
'549300ABE4K96QOCEH37',
'549300CQ9NLEHMRCU505',
'549300DV870NBWY5W279',
'549300DY78U4CMKNHE48',
'549300DYPOFMXOR7XM56',
'549300FOF121DSRG5867',
'549300FR956J8UJDWQ78',
'549300GOF5A5DHWJLV03',
'549300GRXFI7D6PNEA68',
'549300L7YCATGO57ZE10',
'549300LYFYVPUCG6SY25',
'549300NC3SZTETC10349',
'549300NEBDPH0ZXIF850',
'549300Z6OB1D4ZUBD145',
'63540061DPCBNMCGRY22',
'635400CE9HHFB55PEY43',
'635400GQWXFJDCQXW612',
'635400LNHEPZBRNB5D58',
'635400LRAHYBRUZCIH13',
'724500A11NIP5HCVF984',
'815600154F8F91CF6B05',
'8156002070DA4DCBFF31',
'81560027D07F9BDB8436',
'81560038903FF9FA8F80',
'815600522538355AE429',
'8156007395B20763EB44',
'8156009F40F6523F7022',
'815600A32DA05F693F24',
'8EFE15WY4PBBKG6GZI21',
'95980020140005184148',
'9598009X93GQBNHF1L85',
'959800T0J9ADL4GYSS41',
'9695000O0HNLFNR0FB30',
'969500DCEVPV6UIYK220',
'969500QLO3GN6GUB4P61',
'FI6C7E5PBUB3F9K43B44',
'GP5DT10VX1QRQUKVBK64',
'J4CP7MHCXR8DAQMKIL78',
'NNVPP80YIZGEY2314M97',
'RRAN7P32P0W0YY4XQW79',
'SI5RG2M0WQQLZCXKRM20'
)

In [10]:
bgroups = tuple(set(foreign_bg + euro_area_bg))

In [11]:
foreign_entity = tuple(df_mapping_cc.loc[df_mapping_cc['group_id_synth'].isin(foreign_bg), 'lei'].unique())
euro_area_entity = tuple(df_mapping_cc.loc[df_mapping_cc['group_id_synth'].isin(euro_area_bg), 'lei'].unique())

# Probabilities

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
sum(nominal_value) as cleared_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-01-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND lender_id in {foreign_entity}
AND intragroup = 0 
AND central_clearing = 'cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
AND lender_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
GROUP BY business_date, lender_id, security_isin
ORDER BY business_date, lender_id, security_isin
  
"""

df_cleared_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_30904\3925658027.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cleared_lending = pd.read_sql_query(query, cnxn)


In [14]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin,
sum(nominal_value) as cleared_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-01-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND borrower_id in {foreign_entity}
AND intragroup = 0 
AND central_clearing = 'cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
AND borrower_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
GROUP BY business_date, borrower_id, security_isin
ORDER BY business_date, borrower_id, security_isin
  
"""

df_cleared_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_30904\1378425145.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cleared_borrowing = pd.read_sql_query(query, cnxn)


In [15]:
df_cleared = df_cleared_lending.merge(df_cleared_borrowing, on = ['business_date', 'entity_id', 'security_isin'], how = 'outer')

In [16]:
df_cleared['cleared_borrowing'].fillna(0, inplace = True)
df_cleared['cleared_lending'].fillna(0, inplace = True)

In [17]:
df_cleared['cleared_net'] = df_cleared['cleared_lending'] - df_cleared['cleared_borrowing']

In [18]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
sum(nominal_value) as intra_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-01-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND lender_id in {foreign_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
AND lender_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
GROUP BY business_date, lender_id, security_isin
ORDER BY business_date, lender_id, security_isin
  
"""

df_intra_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_30904\4021264915.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_lending = pd.read_sql_query(query, cnxn)


In [19]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
sum(nominal_value) as intra_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-01-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND borrower_id in {foreign_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
AND borrower_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
GROUP BY business_date, borrower_id, security_isin
ORDER BY business_date, borrower_id, security_isin
  
"""

df_intra_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_30904\3698718771.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_borrowing = pd.read_sql_query(query, cnxn)


In [20]:
df_intra = df_intra_lending.merge(df_intra_borrowing, on = ['business_date', 'entity_id', 'security_isin'], how = 'outer')

In [21]:
df_intra['intra_lending'].fillna(0, inplace = True)
df_intra['intra_borrowing'].fillna(0, inplace = True)

In [22]:
df_intra['intra_net'] = df_intra['intra_lending'] - df_intra['intra_borrowing']

In [23]:
df = df_cleared.merge(df_intra, on = ['business_date', 'entity_id', 'security_isin'], how = 'outer')

In [24]:
df.fillna(0, inplace = True)

In [25]:
df.head()

,business_date,entity_id,security_isin,cleared_lending,cleared_borrowing,cleared_net,intra_lending,intra_borrowing,intra_net
0,2021-01-04,2G5BKIC2CB69PRJH1W31,AT0000A0DXC2,2658800.0,0.0,2658800.0,1.143812e+08,0.00,1.143812e+08
1,2021-01-04,2G5BKIC2CB69PRJH1W31,BE0000291972,17894400.0,0.0,17894400.0,0.000000e+00,10577633.72,-1.057763e+07
2,2021-01-04,2G5BKIC2CB69PRJH1W31,BE0000304130,44607500.0,0.0,44607500.0,0.000000e+00,45193589.18,-4.519359e+07
3,2021-01-04,2G5BKIC2CB69PRJH1W31,BE0000320292,12999700.0,0.0,12999700.0,0.000000e+00,14281957.63,-1.428196e+07
4,2021-01-04,2G5BKIC2CB69PRJH1W31,BE0000321308,9436500.0,0.0,9436500.0,0.000000e+00,8601288.76,-8.601289e+06


In [26]:
query = f"""

SELECT business_date, security_isin, 
sum(nominal_value) as hf_activity
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-01-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND lender_id in {euro_area_entity}
AND intragroup = 0
AND central_clearing = 'non-cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
AND s_borrower.sector = 'IF' AND borrower_country_residence = 'KY'
GROUP BY business_date, security_isin
ORDER BY business_date, security_isin
  
"""

df_hf = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_30904\3682314190.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_hf = pd.read_sql_query(query, cnxn)


In [27]:
query = f"""

SELECT business_date, security_isin, 
sum(nominal_value) as total_cleared_volume 
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-01-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 0
AND central_clearing = 'cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, security_isin
ORDER BY business_date, security_isin
  
"""

df_all_clearing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_30904\2902368485.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_all_clearing = pd.read_sql_query(query, cnxn)


In [62]:
# Step 1: Identify chain ISINs per date
# Filter to Euro Area subs and flag ISINs with chain activity
chain_flag = (
    df[(df['intra_borrowing'] > 0)]
    .groupby(['business_date', 'security_isin'])['intra_borrowing']
    .sum()
    .reset_index()
)
chain_flag['is_chain'] = True

In [63]:
df.head()

,business_date,entity_id,security_isin,cleared_lending,cleared_borrowing,cleared_net,intra_lending,intra_borrowing,intra_net
0,2021-01-04,2G5BKIC2CB69PRJH1W31,AT0000A0DXC2,2658800.0,0.0,2658800.0,1.143812e+08,0.00,1.143812e+08
1,2021-01-04,2G5BKIC2CB69PRJH1W31,BE0000291972,17894400.0,0.0,17894400.0,0.000000e+00,10577633.72,-1.057763e+07
2,2021-01-04,2G5BKIC2CB69PRJH1W31,BE0000304130,44607500.0,0.0,44607500.0,0.000000e+00,45193589.18,-4.519359e+07
3,2021-01-04,2G5BKIC2CB69PRJH1W31,BE0000320292,12999700.0,0.0,12999700.0,0.000000e+00,14281957.63,-1.428196e+07
4,2021-01-04,2G5BKIC2CB69PRJH1W31,BE0000321308,9436500.0,0.0,9436500.0,0.000000e+00,8601288.76,-8.601289e+06


In [64]:
# Step 2: Build the panel at ISIN x date level
panel = df_all_clearing.merge(df_hf, on=['business_date', 'security_isin'], how='left')
panel = panel.merge(chain_flag[['business_date', 'security_isin', 'is_chain']],
                    on=['business_date', 'security_isin'], how='left')

panel['hf_activity'] = panel['hf_activity'].fillna(0)
panel['is_chain'] = panel['is_chain'].fillna(False)

In [65]:
panel.head()

,business_date,security_isin,total_cleared_volume,hf_activity,is_chain
0,2021-01-04,AT0000383864,1.533636e+08,0.0,False
1,2021-01-04,AT0000A001X2,1.611936e+08,0.0,True
2,2021-01-04,AT0000A04967,3.617835e+08,0.0,True
3,2021-01-04,AT0000A0DXC2,2.317272e+08,0.0,True
4,2021-01-04,AT0000A0N9A0,7.473100e+07,0.0,False


In [66]:
# Step 3: Daily totals for computing shares
daily_totals = panel.groupby('business_date').agg(
    total_cleared_day=('total_cleared_volume', 'sum'),
    total_hf_day=('hf_activity', 'sum')
).reset_index()

panel = panel.merge(daily_totals, on='business_date')

In [67]:
# Step 4: Shares
panel['hf_share'] = panel['hf_activity'] / panel['total_hf_day']
panel['cleared_share'] = panel['total_cleared_volume'] / panel['total_cleared_day']

# Step 5: Excess HF intensity
panel['excess_hf_intensity'] = np.where(
    panel['cleared_share'] > 0,
    panel['hf_share'] / panel['cleared_share'],
    np.nan
)

In [68]:
# Step 6: The table
# Count based
extensive = panel.groupby('is_chain').agg(
    n_isin_dates=('security_isin', 'count'),
    n_with_hf=('hf_activity', lambda x: (x > 0).sum())
).reset_index()
extensive['prob_hf'] = extensive['n_with_hf'] / extensive['n_isin_dates']

extensive['is_chain'] = extensive['is_chain'].map({True: 'Chain ISINs', False: 'Non-chain ISINs'})

In [69]:
print(extensive.to_string(index=False))

       is_chain  n_isin_dates  n_with_hf  prob_hf
Non-chain ISINs         51518      17486 0.339415
    Chain ISINs        596970     375275 0.628633


In [70]:
print(panel.groupby('is_chain')['security_isin'].nunique())

is_chain
False    1154
True     1386
Name: security_isin, dtype: int64


In [71]:
print(panel.groupby('is_chain')['business_date'].nunique())

is_chain
False    1148
True     1148
Name: business_date, dtype: int64


# share intragroup volume linked to chain

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as if_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND lender_id in {euro_area_entity}
AND intragroup = 0 
AND central_clearing = 'non-cleared'
AND s_borrower.sector = 'IF'
AND security_isin IN {treasuries}
AND gnlcoll = 'SPEC'
GROUP BY business_date, lender_id, residence_group, security_isin
ORDER BY business_date, lender_id, residence_group, security_isin
  
"""

df_if_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_17216\442178056.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_if_lending = pd.read_sql_query(query, cnxn)


In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin,
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as if_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND borrower_id in {euro_area_entity}
AND intragroup = 0 
AND central_clearing = 'non-cleared'
AND s_lender.sector = 'IF'
AND security_isin IN {treasuries}
AND gnlcoll = 'SPEC'
GROUP BY business_date, borrower_id, residence_group, security_isin
ORDER BY business_date, borrower_id, residence_group,security_isin
  
"""

df_if_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_17216\3305393587.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_if_borrowing = pd.read_sql_query(query, cnxn)


In [ ]:
df_if = df_if_lending.merge(df_if_borrowing, on = ['business_date', 'entity_id', 'residence_group', 'security_isin'], how = 'outer')

In [ ]:
df_if['if_borrowing'].fillna(0, inplace = True)
df_if['if_lending'].fillna(0, inplace = True)

In [ ]:
df_if['if_net'] = df_if['if_lending'] - df_if['if_borrowing']

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as intra_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND lender_id in {euro_area_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND security_isin IN {treasuries}
AND gnlcoll = 'SPEC'
GROUP BY business_date, lender_id, residence_group, security_isin
ORDER BY business_date, lender_id, residence_group, security_isin
  
"""

df_intra_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_17216\2444882454.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_lending = pd.read_sql_query(query, cnxn)


In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as intra_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND borrower_id in {euro_area_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND security_isin IN {treasuries}
AND gnlcoll = 'SPEC'
GROUP BY business_date, borrower_id, residence_group, security_isin
ORDER BY business_date, borrower_id, residence_group, security_isin
  
"""

df_intra_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_17216\3430684514.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_borrowing = pd.read_sql_query(query, cnxn)


In [ ]:
df_intra = df_intra_lending.merge(df_intra_borrowing, on = ['business_date', 'entity_id', 'residence_group', 'security_isin'], how = 'outer')

In [ ]:
df_intra['intra_lending'].fillna(0, inplace = True)
df_intra['intra_borrowing'].fillna(0, inplace = True)

In [ ]:
df_intra['intra_net'] = df_intra['intra_lending'] - df_intra['intra_borrowing']

In [ ]:
df = df_if.merge(df_intra, on = ['business_date', 'entity_id', 'residence_group', 'security_isin'], how = 'outer')

In [ ]:
df.fillna(0, inplace = True)

In [ ]:
df.loc[((df['if_borrowing'] > 0) & (df['intra_lending'] > 0)) | ((df['if_lending'] > 0) & (df['intra_borrowing'] > 0)), 'match'] = 'matched'
df['match'].fillna('not matched', inplace = True)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_17216\3475446458.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'matched' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[((df['if_borrowing'] > 0) & (df['intra_lending'] > 0)) | ((df['if_lending'] > 0) & (df['intra_borrowing'] > 0)), 'match'] = 'matched'


In [ ]:
df['chain_intra_to_HF'] = np.minimum(df['intra_borrowing'],
                                df['if_lending'])

df['chain_HF_to_intra'] = np.minimum(df['if_borrowing'],
                                df['intra_lending'])

df['chain_net'] = df['chain_intra_to_HF'] - df['chain_HF_to_intra']

In [ ]:
uk = df[df['residence_group'] == 'UK']

# Lower bounds (size-matched)
chain_vol = uk['chain_net'].abs().sum()
intra_lower = chain_vol / uk['intra_net'].abs().sum()
if_lower = chain_vol / uk['if_net'].abs().sum()

# Upper bounds (co-occurrence)
matched_uk = uk[uk['match'] == 'matched']
intra_upper = matched_uk['intra_net'].abs().sum() / uk['intra_net'].abs().sum()
if_upper = matched_uk['if_net'].abs().sum() / uk['if_net'].abs().sum()

print("=== Intragroup side ===")
print(f"Upper bound (co-occurrence): {intra_upper:.1%}")
print(f"Lower bound (size-matched):  {intra_lower:.1%}")

print("\n=== IF side ===")
print(f"Upper bound (co-occurrence): {if_upper:.1%}")
print(f"Lower bound (size-matched):  {if_lower:.1%}")

=== Intragroup side ===
Upper bound (co-occurrence): 79.3%
Lower bound (size-matched):  57.3%

=== IF side ===
Upper bound (co-occurrence): 97.5%
Lower bound (size-matched):  87.7%


## unconditional matching

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as if_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND intragroup = 0 
AND central_clearing = 'non-cleared'
AND s_borrower.sector = 'IF'
AND security_isin IN {treasuries}
AND gnlcoll = 'SPEC'
GROUP BY business_date, lender_id, residence_group, security_isin
ORDER BY business_date, lender_id, residence_group, security_isin
  
"""

df_if_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_22052\3598298576.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_if_lending = pd.read_sql_query(query, cnxn)


In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin,
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as if_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND intragroup = 0 
AND central_clearing = 'non-cleared'
AND s_lender.sector = 'IF'
AND security_isin IN {treasuries}
AND gnlcoll = 'SPEC'
GROUP BY business_date, borrower_id, residence_group, security_isin
ORDER BY business_date, borrower_id, residence_group,security_isin
  
"""

df_if_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_22052\1057475380.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_if_borrowing = pd.read_sql_query(query, cnxn)


In [ ]:
df_if = df_if_lending.merge(df_if_borrowing, on = ['business_date', 'entity_id', 'residence_group', 'security_isin'], how = 'outer')

In [ ]:
df_if['if_borrowing'].fillna(0, inplace = True)
df_if['if_lending'].fillna(0, inplace = True)

In [ ]:
df_if['if_net'] = df_if['if_lending'] - df_if['if_borrowing']

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as intra_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND lender_id in {euro_area_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND security_isin IN {treasuries}
AND gnlcoll = 'SPEC'
GROUP BY business_date, lender_id, residence_group, security_isin
ORDER BY business_date, lender_id, residence_group, security_isin
  
"""

df_intra_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_22052\2444882454.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_lending = pd.read_sql_query(query, cnxn)


In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as intra_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD'
AND borrower_id in {euro_area_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND security_isin IN {treasuries}
AND gnlcoll = 'SPEC'
GROUP BY business_date, borrower_id, residence_group, security_isin
ORDER BY business_date, borrower_id, residence_group, security_isin
  
"""

df_intra_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_22052\3430684514.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_borrowing = pd.read_sql_query(query, cnxn)


In [ ]:
df_intra = df_intra_lending.merge(df_intra_borrowing, on = ['business_date', 'entity_id', 'residence_group', 'security_isin'], how = 'outer')

In [ ]:
df_intra['intra_lending'].fillna(0, inplace = True)
df_intra['intra_borrowing'].fillna(0, inplace = True)

In [ ]:
df_intra['intra_net'] = df_intra['intra_lending'] - df_intra['intra_borrowing']

In [ ]:
df = df_if.merge(df_intra, on = ['business_date', 'entity_id', 'residence_group', 'security_isin'], how = 'outer')

In [ ]:
df.fillna(0, inplace = True)

In [ ]:
df.loc[((df['if_borrowing'] > 0) & (df['intra_lending'] > 0)) | ((df['if_lending'] > 0) & (df['intra_borrowing'] > 0)), 'match'] = 'matched'
df['match'].fillna('not matched', inplace = True)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_22052\3475446458.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'matched' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[((df['if_borrowing'] > 0) & (df['intra_lending'] > 0)) | ((df['if_lending'] > 0) & (df['intra_borrowing'] > 0)), 'match'] = 'matched'


In [ ]:
df['chain_intra_to_HF'] = np.minimum(df['intra_borrowing'],
                                df['if_lending'])

df['chain_HF_to_intra'] = np.minimum(df['if_borrowing'],
                                df['intra_lending'])

df['chain_net'] = df['chain_intra_to_HF'] - df['chain_HF_to_intra']

In [ ]:
uk = df[df['residence_group'] == 'UK']

# Lower bounds (size-matched)
chain_vol = uk['chain_net'].abs().sum()
intra_lower = chain_vol / uk['intra_net'].abs().sum()
if_lower = chain_vol / uk['if_net'].abs().sum()

# Upper bounds (co-occurrence)
matched_uk = uk[uk['match'] == 'matched']
intra_upper = matched_uk['intra_net'].abs().sum() / uk['intra_net'].abs().sum()
if_upper = matched_uk['if_net'].abs().sum() / uk['if_net'].abs().sum()

print("=== Intragroup side ===")
print(f"Upper bound (co-occurrence): {intra_upper:.1%}")
print(f"Lower bound (size-matched):  {intra_lower:.1%}")

print("\n=== IF side ===")
print(f"Upper bound (co-occurrence): {if_upper:.1%}")
print(f"Lower bound (size-matched):  {if_lower:.1%}")

=== Intragroup side ===
Upper bound (co-occurrence): 79.3%
Lower bound (size-matched):  57.3%

=== IF side ===
Upper bound (co-occurrence): 92.6%
Lower bound (size-matched):  83.3%


# Matching with CCP - EUR

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as cleared_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND lender_id in {foreign_entity}
AND intragroup = 0 
AND central_clearing = 'cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, lender_id, residence_group, security_isin
ORDER BY business_date, lender_id, residence_group, security_isin
  
"""

df_cleared_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_2724\1958244122.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cleared_lending = pd.read_sql_query(query, cnxn)


In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin,
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as cleared_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND borrower_id in {foreign_entity}
AND intragroup = 0 
AND central_clearing = 'cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, borrower_id, residence_group, security_isin
ORDER BY business_date, borrower_id, residence_group, security_isin
  
"""

df_cleared_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_2724\2414576119.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cleared_borrowing = pd.read_sql_query(query, cnxn)


In [ ]:
df_cleared = df_cleared_lending.merge(df_cleared_borrowing, on = ['business_date', 'entity_id', 'residence_group', 'security_isin'], how = 'outer')

In [ ]:
df_cleared['cleared_borrowing'].fillna(0, inplace = True)
df_cleared['cleared_lending'].fillna(0, inplace = True)

In [ ]:
df_cleared['cleared_net'] = df_cleared['cleared_lending'] - df_cleared['cleared_borrowing']

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as intra_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND lender_id in {foreign_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, lender_id, residence_group, security_isin
ORDER BY business_date, lender_id, residence_group, security_isin
  
"""

df_intra_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_2724\3572604576.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_lending = pd.read_sql_query(query, cnxn)


In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as intra_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND borrower_id in {foreign_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, borrower_id, residence_group, security_isin
ORDER BY business_date, borrower_id, residence_group, security_isin
  
"""

df_intra_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_17216\3713498267.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_borrowing = pd.read_sql_query(query, cnxn)


In [ ]:
df_intra = df_intra_lending.merge(df_intra_borrowing, on = ['business_date', 'entity_id', 'residence_group', 'security_isin'], how = 'outer')

In [ ]:
df_intra['intra_lending'].fillna(0, inplace = True)
df_intra['intra_borrowing'].fillna(0, inplace = True)

In [ ]:
df_intra['intra_net'] = df_intra['intra_lending'] - df_intra['intra_borrowing']

In [ ]:
df = df_cleared.merge(df_intra, on = ['business_date', 'entity_id', 'residence_group', 'security_isin'], how = 'outer')

In [ ]:
df.fillna(0, inplace = True)

In [ ]:
df.loc[((df['cleared_borrowing'] > 0) & (df['intra_lending'] > 0)) | ((df['cleared_lending'] > 0) & (df['intra_borrowing'] > 0)), 'match'] = 'matched'
df['match'].fillna('not matched', inplace = True)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_17216\3724210190.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'matched' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[((df['cleared_borrowing'] > 0) & (df['intra_lending'] > 0)) | ((df['cleared_lending'] > 0) & (df['intra_borrowing'] > 0)), 'match'] = 'matched'


In [ ]:
df['chain_intra_to_ccp'] = np.minimum(df['intra_borrowing'],
                                df['cleared_lending'])

df['chain_ccp_to_intra'] = np.minimum(df['cleared_borrowing'],
                                df['intra_lending'])

df['chain_net'] = df['chain_intra_to_ccp'] - df['chain_ccp_to_intra']

In [ ]:
uk = df[df['residence_group'] == 'Euro Area']

# Lower bounds (size-matched)
chain_vol = uk['chain_net'].abs().sum()
intra_lower = chain_vol / uk['intra_net'].abs().sum()
cleared_lower = chain_vol / uk['cleared_net'].abs().sum()

# Upper bounds (co-occurrence)
matched_uk = uk[uk['match'] == 'matched']
intra_upper = matched_uk['intra_net'].abs().sum() / uk['intra_net'].abs().sum()
cleared_upper = matched_uk['cleared_net'].abs().sum() / uk['cleared_net'].abs().sum()

print("=== Intragroup side ===")
print(f"Upper bound (co-occurrence): {intra_upper:.1%}")
print(f"Lower bound (size-matched):  {intra_lower:.1%}")

print("\n=== Cleared side ===")
print(f"Upper bound (co-occurrence): {cleared_upper:.1%}")
print(f"Lower bound (size-matched):  {cleared_lower:.1%}")

=== Intragroup side ===
Upper bound (co-occurrence): 63.8%
Lower bound (size-matched):  36.7%

=== Cleared side ===
Upper bound (co-occurrence): 50.7%
Lower bound (size-matched):  30.7%


## unconditional matching

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as cleared_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 0 
AND central_clearing = 'cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, lender_id, residence_group, security_isin
ORDER BY business_date, lender_id, residence_group, security_isin
  
"""

df_cleared_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_22052\2718059570.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cleared_lending = pd.read_sql_query(query, cnxn)


In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin,
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as cleared_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 0 
AND central_clearing = 'cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, borrower_id, residence_group, security_isin
ORDER BY business_date, borrower_id, residence_group, security_isin
  
"""

df_cleared_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_22052\1213816397.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cleared_borrowing = pd.read_sql_query(query, cnxn)


In [ ]:
df_cleared = df_cleared_lending.merge(df_cleared_borrowing, on = ['business_date', 'entity_id', 'residence_group', 'security_isin'], how = 'outer')

In [ ]:
df_cleared['cleared_borrowing'].fillna(0, inplace = True)
df_cleared['cleared_lending'].fillna(0, inplace = True)

In [ ]:
df_cleared['cleared_net'] = df_cleared['cleared_lending'] - df_cleared['cleared_borrowing']

In [ ]:
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, 
CASE
    WHEN s.lender_country_residence IN ('US') THEN 'US'
    WHEN s.lender_country_residence IN ('GB') THEN 'UK'
    WHEN s.lender_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as intra_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND lender_id in {foreign_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, lender_id, residence_group, security_isin
ORDER BY business_date, lender_id, residence_group, security_isin
  
"""

df_intra_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_22052\3572604576.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_lending = pd.read_sql_query(query, cnxn)


In [ ]:
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, 
CASE
    WHEN s.borrower_country_residence IN ('US') THEN 'US'
    WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
    WHEN s.borrower_country_residence IN
         ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
      THEN 'Euro Area'
    ELSE 'Other'
  END AS residence_group,
sum(nominal_value) as intra_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND borrower_id in {foreign_entity}
AND intragroup = 1
AND central_clearing = 'non-cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, borrower_id, residence_group, security_isin
ORDER BY business_date, borrower_id, residence_group, security_isin
  
"""

df_intra_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_22052\3713498267.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_borrowing = pd.read_sql_query(query, cnxn)


In [ ]:
df_intra = df_intra_lending.merge(df_intra_borrowing, on = ['business_date', 'entity_id', 'residence_group', 'security_isin'], how = 'outer')

In [ ]:
df_intra['intra_lending'].fillna(0, inplace = True)
df_intra['intra_borrowing'].fillna(0, inplace = True)

In [ ]:
df_intra['intra_net'] = df_intra['intra_lending'] - df_intra['intra_borrowing']

In [ ]:
df = df_cleared.merge(df_intra, on = ['business_date', 'entity_id', 'residence_group', 'security_isin'], how = 'outer')

In [ ]:
df.fillna(0, inplace = True)

In [ ]:
df.loc[((df['cleared_borrowing'] > 0) & (df['intra_lending'] > 0)) | ((df['cleared_lending'] > 0) & (df['intra_borrowing'] > 0)), 'match'] = 'matched'
df['match'].fillna('not matched', inplace = True)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_22052\3724210190.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'matched' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[((df['cleared_borrowing'] > 0) & (df['intra_lending'] > 0)) | ((df['cleared_lending'] > 0) & (df['intra_borrowing'] > 0)), 'match'] = 'matched'


In [ ]:
df['chain_intra_to_ccp'] = np.minimum(df['intra_borrowing'],
                                df['cleared_lending'])

df['chain_ccp_to_intra'] = np.minimum(df['cleared_borrowing'],
                                df['intra_lending'])

df['chain_net'] = df['chain_intra_to_ccp'] - df['chain_ccp_to_intra']

In [ ]:
uk = df[df['residence_group'] == 'Euro Area']

# Lower bounds (size-matched)
chain_vol = uk['chain_net'].abs().sum()
intra_lower = chain_vol / uk['intra_net'].abs().sum()
cleared_lower = chain_vol / uk['cleared_net'].abs().sum()

# Upper bounds (co-occurrence)
matched_uk = uk[uk['match'] == 'matched']
intra_upper = matched_uk['intra_net'].abs().sum() / uk['intra_net'].abs().sum()
cleared_upper = matched_uk['cleared_net'].abs().sum() / uk['cleared_net'].abs().sum()

print("=== Intragroup side ===")
print(f"Upper bound (co-occurrence): {intra_upper:.1%}")
print(f"Lower bound (size-matched):  {intra_lower:.1%}")

print("\n=== Cleared side ===")
print(f"Upper bound (co-occurrence): {cleared_upper:.1%}")
print(f"Lower bound (size-matched):  {cleared_lower:.1%}")

=== Intragroup side ===
Upper bound (co-occurrence): 63.8%
Lower bound (size-matched):  36.7%

=== Cleared side ===
Upper bound (co-occurrence): 13.8%
Lower bound (size-matched):  8.4%


## Pass-through regressions and USD descriptive evidence (appended)

These cells are self-contained. They re-query the legs they need with fresh names, so they do not depend on which `df` is in memory after the blocks above. The USD pass-through and the descriptive evidence cover euro area groups' UK branches facing investment funds (`IF`) in Treasuries. The EUR pass-through covers foreign groups' euro area subsidiaries on the intragroup-to-CCP leg. Each block writes a CSV to the `Data` folder for the Stata regressions and graphs.


In [12]:
# ============================================================
# WITHIN-ENTITY PASS-THROUGH (USD) AND USD DESCRIPTIVE EVIDENCE
# USD chain actors are euro area groups' UK branches. Hedge funds are
# Cayman-resident investment funds. Self-contained, re-queries with fresh
# names. Saves Data\pt_usd.csv (pass-through panel) and Data\desc_usd.csv
# (daily net positions and gross financing).
# ============================================================
q_if_l = f"""
SELECT business_date, lender_id as entity_id, security_isin,
CASE WHEN s.lender_country_residence IN ('US') THEN 'US'
     WHEN s.lender_country_residence IN ('GB') THEN 'UK'
     WHEN s.lender_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES') THEN 'Euro Area'
     ELSE 'Other' END AS residence_group,
sum(nominal_value) as if_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD' AND lender_id in {euro_area_entity} AND intragroup = 0
AND central_clearing = 'non-cleared' AND s_borrower.sector = 'IF' AND borrower_country_residence = 'KY'
AND security_isin IN {treasuries} AND gnlcoll = 'SPEC'
GROUP BY business_date, lender_id, residence_group, security_isin
"""
q_if_b = f"""
SELECT business_date, borrower_id as entity_id, security_isin,
CASE WHEN s.borrower_country_residence IN ('US') THEN 'US'
     WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
     WHEN s.borrower_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES') THEN 'Euro Area'
     ELSE 'Other' END AS residence_group,
sum(nominal_value) as if_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD' AND borrower_id in {euro_area_entity} AND intragroup = 0
AND central_clearing = 'non-cleared' AND s_lender.sector = 'IF' AND lender_country_residence = 'KY'
AND security_isin IN {treasuries} AND gnlcoll = 'SPEC'
GROUP BY business_date, borrower_id, residence_group, security_isin
"""
q_in_l = f"""
SELECT business_date, lender_id as entity_id, security_isin,
CASE WHEN s.lender_country_residence IN ('US') THEN 'US'
     WHEN s.lender_country_residence IN ('GB') THEN 'UK'
     WHEN s.lender_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES') THEN 'Euro Area'
     ELSE 'Other' END AS residence_group,
sum(nominal_value) as intra_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD' AND lender_id in {euro_area_entity} AND intragroup = 1
AND central_clearing = 'non-cleared' AND security_isin IN {treasuries} AND gnlcoll = 'SPEC'
GROUP BY business_date, lender_id, residence_group, security_isin
"""
q_in_b = f"""
SELECT business_date, borrower_id as entity_id, security_isin,
CASE WHEN s.borrower_country_residence IN ('US') THEN 'US'
     WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
     WHEN s.borrower_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES') THEN 'Euro Area'
     ELSE 'Other' END AS residence_group,
sum(nominal_value) as intra_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'USD' AND borrower_id in {euro_area_entity} AND intragroup = 1
AND central_clearing = 'non-cleared' AND security_isin IN {treasuries} AND gnlcoll = 'SPEC'
GROUP BY business_date, borrower_id, residence_group, security_isin
"""
usd_if_l = pd.read_sql_query(q_if_l, cnxn)
usd_if_b = pd.read_sql_query(q_if_b, cnxn)
usd_in_l = pd.read_sql_query(q_in_l, cnxn)
usd_in_b = pd.read_sql_query(q_in_b, cnxn)

keys = ['business_date', 'entity_id', 'residence_group', 'security_isin']
df_usd = (usd_if_l.merge(usd_if_b, on=keys, how='outer')
                  .merge(usd_in_l, on=keys, how='outer')
                  .merge(usd_in_b, on=keys, how='outer'))
for c in ['if_lending', 'if_borrowing', 'intra_lending', 'intra_borrowing']:
    df_usd[c] = df_usd[c].fillna(0)

# Pass-through panel: UK branches only. Gross volumes for co-movement, nets for offset.
usd = df_usd[df_usd['residence_group'] == 'UK'].copy()
usd['intra_vol'] = (usd['intra_lending'] + usd['intra_borrowing']) / 1e9
usd['hf_vol']    = (usd['if_lending'] + usd['if_borrowing']) / 1e9
usd['intra_net'] = (usd['intra_lending'] - usd['intra_borrowing']) / 1e9
usd['if_net']    = (usd['if_lending'] - usd['if_borrowing']) / 1e9
usd[['business_date', 'entity_id', 'security_isin', 'intra_vol', 'hf_vol', 'intra_net', 'if_net']].to_csv('Data\\pt_usd.csv', index=False)

# Descriptive: UK-branch positions vis-a-vis hedge funds, daily, USD bn.
# hf_net > 0 means the UK branch is a net lender of cash to hedge funds (net
# receiver of Treasuries). intra_net is the offsetting net intragroup position.
uk = df_usd[df_usd['residence_group'] == 'UK'].copy()
uk['hf_gross']  = uk['if_lending'] + uk['if_borrowing']
uk['hf_net']    = uk['if_lending'] - uk['if_borrowing']
uk['intra_net'] = uk['intra_lending'] - uk['intra_borrowing']
daily = uk.groupby('business_date', as_index=False).agg(
    hf_gross=('hf_gross', 'sum'),
    hf_net=('hf_net', 'sum'),
    intra_net=('intra_net', 'sum'))
daily['hf_gross_bn']  = daily['hf_gross'] / 1e9
daily['hf_net_bn']    = daily['hf_net'] / 1e9
daily['intra_net_bn'] = daily['intra_net'] / 1e9
daily[['business_date', 'hf_gross_bn', 'hf_net_bn', 'intra_net_bn']].to_csv('Data\\desc_usd.csv', index=False)


C:\Users\hermesf\AppData\Local\Temp\ipykernel_47732\3357555944.py:66: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  usd_if_l = pd.read_sql_query(q_if_l, cnxn)
C:\Users\hermesf\AppData\Local\Temp\ipykernel_47732\3357555944.py:67: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  usd_if_b = pd.read_sql_query(q_if_b, cnxn)
C:\Users\hermesf\AppData\Local\Temp\ipykernel_47732\3357555944.py:68: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  usd_in_l = pd.read_sql_query(q_in_l, cnxn)
C:\Users\hermesf\AppData\Local\Temp\ipykernel_47732\

In [13]:
# ============================================================
# WITHIN-ENTITY PASS-THROUGH (EUR)
# EUR chain actors are foreign groups' euro area subsidiaries, matched on the
# intragroup-to-CCP leg. Self-contained, re-queries with fresh names.
# Saves Data\pt_eur.csv.
# ============================================================
q_cl_l = f"""
SELECT business_date, lender_id as entity_id, security_isin,
CASE WHEN s.lender_country_residence IN ('US') THEN 'US'
     WHEN s.lender_country_residence IN ('GB') THEN 'UK'
     WHEN s.lender_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES') THEN 'Euro Area'
     ELSE 'Other' END AS residence_group,
sum(nominal_value) as cleared_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND lender_id in {foreign_entity} AND intragroup = 0
AND central_clearing = 'cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC' AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC' AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, lender_id, residence_group, security_isin
"""
q_cl_b = f"""
SELECT business_date, borrower_id as entity_id, security_isin,
CASE WHEN s.borrower_country_residence IN ('US') THEN 'US'
     WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
     WHEN s.borrower_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES') THEN 'Euro Area'
     ELSE 'Other' END AS residence_group,
sum(nominal_value) as cleared_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND borrower_id in {foreign_entity} AND intragroup = 0
AND central_clearing = 'cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC' AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC' AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, borrower_id, residence_group, security_isin
"""
q_in_l = f"""
SELECT business_date, lender_id as entity_id, security_isin,
CASE WHEN s.lender_country_residence IN ('US') THEN 'US'
     WHEN s.lender_country_residence IN ('GB') THEN 'UK'
     WHEN s.lender_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES') THEN 'Euro Area'
     ELSE 'Other' END AS residence_group,
sum(nominal_value) as intra_lending
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND lender_id in {foreign_entity} AND intragroup = 1
AND central_clearing = 'non-cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC' AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC' AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, lender_id, residence_group, security_isin
"""
q_in_b = f"""
SELECT business_date, borrower_id as entity_id, security_isin,
CASE WHEN s.borrower_country_residence IN ('US') THEN 'US'
     WHEN s.borrower_country_residence IN ('GB') THEN 'UK'
     WHEN s.borrower_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES') THEN 'Euro Area'
     ELSE 'Other' END AS residence_group,
sum(nominal_value) as intra_borrowing
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND borrower_id in {foreign_entity} AND intragroup = 1
AND central_clearing = 'non-cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC' AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC' AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, borrower_id, residence_group, security_isin
"""
eur_cl_l = pd.read_sql_query(q_cl_l, cnxn)
eur_cl_b = pd.read_sql_query(q_cl_b, cnxn)
eur_in_l = pd.read_sql_query(q_in_l, cnxn)
eur_in_b = pd.read_sql_query(q_in_b, cnxn)

keys = ['business_date', 'entity_id', 'residence_group', 'security_isin']
df_eur = (eur_cl_l.merge(eur_cl_b, on=keys, how='outer')
                  .merge(eur_in_l, on=keys, how='outer')
                  .merge(eur_in_b, on=keys, how='outer'))
for c in ['cleared_lending', 'cleared_borrowing', 'intra_lending', 'intra_borrowing']:
    df_eur[c] = df_eur[c].fillna(0)

# Euro area subs only. Gross volumes for co-movement, nets for offset.
eur = df_eur[df_eur['residence_group'] == 'Euro Area'].copy()
eur['intra_vol']   = (eur['intra_lending'] + eur['intra_borrowing']) / 1e9
eur['cleared_vol'] = (eur['cleared_lending'] + eur['cleared_borrowing']) / 1e9
eur['intra_net']   = (eur['intra_lending'] - eur['intra_borrowing']) / 1e9
eur['cleared_net'] = (eur['cleared_lending'] - eur['cleared_borrowing']) / 1e9
eur[['business_date', 'entity_id', 'security_isin', 'intra_vol', 'cleared_vol', 'intra_net', 'cleared_net']].to_csv('Data\\pt_eur.csv', index=False)

# Descriptive: euro area subs' net positions on the cleared (CCP) leg vs
# intragroup, daily, EUR bn. cleared_net > 0 means net lender of cash in cleared
# (net receiver of bonds, i.e. sourcing); intra_net is the offsetting position.
daily_eur = eur.groupby('business_date', as_index=False).agg(
    cleared_gross_bn=('cleared_vol', 'sum'),
    cleared_net_bn=('cleared_net', 'sum'),
    intra_net_bn=('intra_net', 'sum'))
daily_eur.to_csv('Data\\desc_eur.csv', index=False)


C:\Users\hermesf\AppData\Local\Temp\ipykernel_47732\3210420622.py:91: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  eur_cl_l = pd.read_sql_query(q_cl_l, cnxn)
C:\Users\hermesf\AppData\Local\Temp\ipykernel_47732\3210420622.py:92: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  eur_cl_b = pd.read_sql_query(q_cl_b, cnxn)
C:\Users\hermesf\AppData\Local\Temp\ipykernel_47732\3210420622.py:93: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  eur_in_l = pd.read_sql_query(q_in_l, cnxn)
C:\Users\hermesf\AppData\Local\Temp\ipykernel_47732\